# Lab 3.4 &mdash; Positional Encoding

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 2 &middot; Module 3 &mdash; Why Transformers?**

### What you'll do
- Understand why attention alone is order-blind
- Implement sinusoidal positional encodings
- Confirm each position gets a distinct vector

> **How this lab works (near-real):** these labs run **real Hugging Face Transformers** locally on CPU. Read the **Concept**, fill the real `___` blanks in **Build it** (real tokenizer / model / decoding calls), **Run it for real** to see the **actual model output**, note **What to notice**, then finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is real model output you can read. The genuine maths (attention, positional encoding, cosine) you still compute **by hand** &mdash; that is real mechanics, not a stub.

> **Models:** small, CPU-friendly models from the HF hub &mdash; `distilbert-base-uncased` (tokenizer / fill-mask), `sentence-transformers/all-MiniLM-L6-v2` (embeddings), `prajjwal1/bert-tiny` (attention), `distilgpt2` (generation). First use downloads the weights (needs network), then they are cached. The hosted "GPT API" path uses `ChatGroq` (`GROQ_API_KEY` in `.env`).

**Reference:** [Module 3 slides &mdash; Positional encoding](../../presentation/day2-module3-why-transformers.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 3 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("USE_TF", "0")                 # these labs are torch-only; skip the TF backend
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # mute TensorFlow's C++ INFO/WARNING startup noise
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY etc. (used by the text-gen lab)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-03-04")
os.makedirs(WORK, exist_ok=True)
print("WORK:", WORK)
print("Real Hugging Face models load from the hub on first use (one-time download, then cached).")

## Concept
Attention treats a sentence as a **set** &mdash; it has no built-in sense of order. So we **add** a
**positional encoding** to each token's embedding. The classic recipe uses sines and cosines of
different frequencies:
`PE(pos, 2i) = sin(pos / 10000^(2i/d))`, `PE(pos, 2i+1) = cos(pos / 10000^(2i/d))`.
This is the real formula from the original transformer paper.

> The plot needs `matplotlib` (already in the lab venv).

## Build it
Fill in the **angle** and the **sin/cos** assignments.

In [ ]:
import numpy as np
def positional_encoding(seq_len, d_model):
    pos = np.arange(seq_len)[:, None]
    i = np.arange(d_model)[None, :]
    angle = ___   # TODO: pos / (10000 ** (2*(i//2) / d_model))
    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = ___   # TODO: sin of the even-index angles
    pe[:, 1::2] = ___   # TODO: cos of the odd-index angles
    return pe

## Run it for real
Build a positional-encoding matrix and plot it.

In [ ]:
try:
    import numpy as np
    PE = positional_encoding(10, 16)
    print("shape:", PE.shape)
    print("position 0:", np.round(PE[0], 2))
    print("position 1:", np.round(PE[1], 2))
    import matplotlib.pyplot as plt
    plt.imshow(PE, cmap="RdBu", aspect="auto"); plt.xlabel("dimension"); plt.ylabel("position")
    plt.title("Sinusoidal positional encoding"); plt.colorbar(); plt.tight_layout()
    plt.savefig(WORK + "/positional_encoding.png", dpi=90); plt.show()
    print("saved:", WORK + "/positional_encoding.png")
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__, "--", e)

## What to notice
- Position 0 starts `sin=0, cos=1` &mdash; every position gets a **distinct** vector.
- The plot shows **stripes**: low dimensions oscillate fast, high dimensions slow &mdash; a multi-scale "clock" the model reads as position.
- Every value is in `[-1, 1]`, so it can be **added** to token embeddings without dominating them.

## Your turn (open task &mdash; no grader)
Change `seq_len` and `d_model` and re-plot. Then compute the cosine similarity between the
encodings of positions 0&1 vs 0&9 &mdash; does "closer in the sentence" mean "more similar encoding"?
A "good" answer: you can point to the plot and explain how the model could recover *distance* between
two positions from these vectors.

---
### Key takeaway
Embedding + positional encoding is what actually enters a transformer block. Now the model knows both *what* and *where*.

[&#8592; All Module 3 labs](./index.html) &nbsp;&middot;&nbsp; [Module 3 slides](../../presentation/day2-module3-why-transformers.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>